# Convolutional AutoEncoder for Spatio-Temporal Data Compression**Approach**: Each timestep is interpolated onto a regular 2D grid and treated as a multi-channel image. A Conv2D autoencoder compresses each timestep image to a compact latent vector.**Three model configurations** are trained and compared:| Model | Encoder Channels | Latent Dim | Grid ||-------|-----------------|------------|------|| **Base** | 4 → 16 → 32 → 64 → 128 | 32 | 32 x 128 || **Medium** | 4 → 32 → 64 → 128 → 256 | 64 | 32 x 128 || **Large** | 4 → 32 → 64 → 128 → 256 | 128 | 32 x 128 |**Key difference from Linear AE**: Latent codes are per-timestep (300 vectors), not per-spatial-point (26,397 vectors), so latent storage is negligible.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session
# Data path: /kaggle/input/datasets/maheshsadupalli/ml-test-loader-original-data-csv/ML_test_loader_original_data.csv


In [ ]:
"""
Convolutional AutoEncoder: Utilities, Models, and Training Functions

Handles grid interpolation from unstructured mesh, Conv2D models,
training, evaluation, and reverse interpolation for mesh-level metrics.
"""

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import pyarrow.csv as pv
from scipy.spatial import Delaunay, cKDTree
from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator
from scipy.interpolate import RegularGridInterpolator
from torcheval.metrics import PeakSignalNoiseRatio
from torchmetrics.image import StructuralSimilarityIndexMeasure
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt
import time
import os
import json


# Dataset

class GridImageDataset(Dataset):
    """
    Converts unstructured mesh data to regular grid images for Conv2D.

    Each sample is one timestep as a multi-channel image:
    (num_vars, grid_h, grid_w) = (4, 32, 128).

    Pipeline:
        Unstructured mesh (26,397 points) --> interpolate --> grid (32 x 128)
        One grid image per timestep --> 300 samples total.
    """

    def __init__(self, filepath, grid_h=32, grid_w=128):
        print("Loading dataset from: {}".format(filepath))

        read_options = pv.ReadOptions(
            column_names=['x', 'y', 'z', 't', 'Vx', 'Vy', 'Pressure', 'TKE']
        )
        table = pv.read_csv(filepath, read_options=read_options)
        data = table.to_pandas()
        data = data.sort_values(['x', 'y', 'z', 't']).reset_index(drop=True)

        self.time_values = np.sort(data['t'].unique()).astype(np.float32)
        self.num_timesteps = len(self.time_values)
        self.num_vars = 4
        self.var_names = ['Vx', 'Vy', 'Pressure', 'TKE']
        self.grid_h = grid_h
        self.grid_w = grid_w

        fields = data[['Vx', 'Vy', 'Pressure', 'TKE']].values.astype(np.float32)
        self.num_points = len(data) // self.num_timesteps

        # Original mesh coordinates (same for all timesteps)
        self.mesh_coords = data[['x', 'y']].values[::self.num_timesteps].astype(np.float32)

        # Min-max normalization
        self.field_min = fields.min(axis=0)
        self.field_max = fields.max(axis=0)
        self.field_range = self.field_max - self.field_min
        self.field_range[self.field_range == 0] = 1.0

        # Reshape to (T, N, V) and normalize
        fields_3d = fields.reshape(self.num_points, self.num_timesteps, self.num_vars)
        fields_3d = fields_3d.transpose(1, 0, 2)  # (T, N, V)
        self.mesh_fields_norm = (fields_3d - self.field_min) / self.field_range

        print("Detected {} spatial points across {} timesteps".format(
            self.num_points, self.num_timesteps))

        # Regular grid
        x_min, x_max = self.mesh_coords[:, 0].min(), self.mesh_coords[:, 0].max()
        y_min, y_max = self.mesh_coords[:, 1].min(), self.mesh_coords[:, 1].max()
        self.grid_x = np.linspace(x_min, x_max, grid_w).astype(np.float32)
        self.grid_y = np.linspace(y_min, y_max, grid_h).astype(np.float32)
        self.gx, self.gy = np.meshgrid(self.grid_x, self.grid_y)

        print("Interpolating to regular grid ({} x {})...".format(grid_h, grid_w))

        # Delaunay triangulation (computed once)
        tri = Delaunay(self.mesh_coords)

        # Precompute NaN mask and nearest-neighbor indices
        dummy = LinearNDInterpolator(tri, np.ones(self.num_points))
        dummy_vals = dummy(self.gx, self.gy)
        self.nan_mask = np.isnan(dummy_vals)

        if self.nan_mask.any():
            nan_xy = np.column_stack([
                self.gx[self.nan_mask], self.gy[self.nan_mask]
            ])
            tree = cKDTree(self.mesh_coords)
            _, self.nearest_idx = tree.query(nan_xy)
            print("  {} grid points inside boundary filled via nearest neighbor".format(
                self.nan_mask.sum()))

        # Interpolate all timesteps to grid
        grid_data = np.zeros(
            (self.num_timesteps, self.num_vars, grid_h, grid_w),
            dtype=np.float32
        )

        for t in range(self.num_timesteps):
            for v in range(self.num_vars):
                values = self.mesh_fields_norm[t, :, v]
                lin = LinearNDInterpolator(tri, values)
                gv = lin(self.gx, self.gy)

                if self.nan_mask.any():
                    gv[self.nan_mask] = values[self.nearest_idx]

                grid_data[t, v] = gv

            if (t + 1) % 50 == 0:
                print("  Interpolated {}/{} timesteps".format(
                    t + 1, self.num_timesteps))

        self.grid_data = torch.FloatTensor(grid_data)

        print("Grid dataset ready: {} samples of shape ({}, {}, {})".format(
            self.num_timesteps, self.num_vars, grid_h, grid_w))

    def __len__(self):
        return self.num_timesteps

    def __getitem__(self, idx):
        x = self.grid_data[idx]
        return x, x

    def grid_to_mesh(self, grid_predictions):
        """
        Interpolate grid predictions back to original mesh points.

        Args:
            grid_predictions: numpy array, shape (T, V, H, W) or (V, H, W)

        Returns:
            numpy array, shape (T, N, V) or (N, V)
        """
        single = grid_predictions.ndim == 3
        if single:
            grid_predictions = grid_predictions[np.newaxis]

        T = grid_predictions.shape[0]
        mesh_preds = np.zeros(
            (T, self.num_points, self.num_vars), dtype=np.float32
        )
        query_pts = np.column_stack([
            self.mesh_coords[:, 1], self.mesh_coords[:, 0]
        ])

        for t in range(T):
            for v in range(self.num_vars):
                interp = RegularGridInterpolator(
                    (self.grid_y, self.grid_x),
                    grid_predictions[t, v],
                    method='linear',
                    bounds_error=False,
                    fill_value=None
                )
                mesh_preds[t, :, v] = interp(query_pts)

        return mesh_preds[0] if single else mesh_preds

    def get_normalization_params(self):
        return {
            'field_min': self.field_min.tolist(),
            'field_max': self.field_max.tolist(),
            'field_range': self.field_range.tolist(),
            'num_timesteps': self.num_timesteps,
            'num_points': self.num_points,
            'num_vars': self.num_vars,
            'grid_h': self.grid_h,
            'grid_w': self.grid_w,
        }


# Models

class ConvEncoder(nn.Module):
    """Conv2D encoder with stride-2 downsampling and a linear bottleneck."""

    def __init__(self, in_channels, channel_list, latent_dim):
        super().__init__()
        conv_layers = []
        prev_ch = in_channels
        for ch in channel_list:
            conv_layers.extend([
                nn.Conv2d(prev_ch, ch, kernel_size=3, stride=2, padding=1),
                nn.BatchNorm2d(ch),
                nn.LeakyReLU(0.1),
            ])
            prev_ch = ch
        self.conv = nn.Sequential(*conv_layers)
        self.latent_dim = latent_dim
        # Flatten dim computed dynamically in first forward pass
        self._flatten_dim = None
        self._fc = None

    def _build_fc(self, x):
        self._flatten_dim = x.shape[1] * x.shape[2] * x.shape[3]
        self._fc = nn.Linear(self._flatten_dim, self.latent_dim).to(x.device)

    def forward(self, x):
        x = self.conv(x)
        if self._fc is None:
            self._build_fc(x)
        x = x.view(x.size(0), -1)
        return self._fc(x)


class ConvDecoder(nn.Module):
    """Conv2D decoder with stride-2 upsampling from a linear bottleneck."""

    def __init__(self, out_channels, channel_list, latent_dim,
                 bottleneck_shape):
        super().__init__()
        self.bottleneck_shape = bottleneck_shape  # (C, H, W)
        flatten_dim = bottleneck_shape[0] * bottleneck_shape[1] * bottleneck_shape[2]
        self.fc = nn.Linear(latent_dim, flatten_dim)

        conv_layers = []
        reversed_ch = list(reversed(channel_list))
        for i in range(len(reversed_ch) - 1):
            conv_layers.extend([
                nn.ConvTranspose2d(
                    reversed_ch[i], reversed_ch[i + 1],
                    kernel_size=3, stride=2, padding=1, output_padding=1
                ),
                nn.BatchNorm2d(reversed_ch[i + 1]),
                nn.LeakyReLU(0.1),
            ])
        # Final layer: back to original channels, no BN or activation
        conv_layers.append(
            nn.ConvTranspose2d(
                reversed_ch[-1], out_channels,
                kernel_size=3, stride=2, padding=1, output_padding=1
            )
        )
        self.conv = nn.Sequential(*conv_layers)

    def forward(self, z):
        x = self.fc(z)
        x = x.view(x.size(0), *self.bottleneck_shape)
        return self.conv(x)


class ConvAutoEncoder(nn.Module):
    """
    Convolutional autoencoder for spatio-temporal grid data.

    Encodes each timestep image (4, H, W) into a compact latent vector.
    """

    def __init__(self, in_channels, channel_list, latent_dim, input_shape):
        super().__init__()
        self.latent_dim = latent_dim

        # Compute bottleneck shape by running a dummy forward
        dummy = torch.zeros(1, in_channels, *input_shape)
        conv_layers = []
        prev_ch = in_channels
        for ch in channel_list:
            conv_layers.append(
                nn.Conv2d(prev_ch, ch, kernel_size=3, stride=2, padding=1)
            )
            prev_ch = ch
        temp_conv = nn.Sequential(*conv_layers)
        with torch.no_grad():
            dummy_out = temp_conv(dummy)
        bottleneck_shape = (
            channel_list[-1], dummy_out.shape[2], dummy_out.shape[3]
        )

        self.encoder = ConvEncoder(in_channels, channel_list, latent_dim)
        self.decoder = ConvDecoder(
            in_channels, channel_list, latent_dim, bottleneck_shape
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)


# Model Configurations

CONV_AE_CONFIGS = {
    'base': {
        'channel_list': [16, 32, 64, 128],
        'latent_dim': 32,
    },
    'medium': {
        'channel_list': [32, 64, 128, 256],
        'latent_dim': 64,
    },
    'large': {
        'channel_list': [32, 64, 128, 256],
        'latent_dim': 128,
    },
}


def create_conv_autoencoder(size, input_shape, in_channels=4):
    """Instantiate a ConvAutoEncoder from a named configuration."""
    cfg = CONV_AE_CONFIGS[size]
    return ConvAutoEncoder(
        in_channels=in_channels,
        channel_list=cfg['channel_list'],
        latent_dim=cfg['latent_dim'],
        input_shape=input_shape,
    )


# Metrics

def compute_psnr_ssim(predictions, targets, device):
    """Compute PSNR (dB) and SSIM for reconstruction quality."""
    predictions = predictions.to(device)
    targets = targets.to(device)

    psnr_metric = PeakSignalNoiseRatio().to(device)
    psnr_metric.update(predictions, targets)
    psnr = psnr_metric.compute().item()

    pred_ssim = predictions.unsqueeze(0).unsqueeze(0)
    target_ssim = targets.unsqueeze(0).unsqueeze(0)
    ssim_metric = StructuralSimilarityIndexMeasure(
        gaussian_kernel=False, kernel_size=1
    ).to(device)
    ssim_metric.update(pred_ssim, target_ssim)
    ssim = ssim_metric.compute().item()

    return psnr, ssim


def compute_relative_error(predictions, targets):
    """Compute relative L2 norm error as a percentage."""
    error_norm = torch.norm(predictions - targets)
    target_norm = torch.norm(targets)
    return (error_norm / target_norm * 100).item()


def compute_compression_ratio(model, dataset):
    """
    Compression ratio for Conv AE.

    Compressed = model weights + latent codes (one per timestep).
    """
    total_params = sum(p.numel() for p in model.parameters())
    model_bytes = total_params * 4
    latent_bytes = dataset.num_timesteps * model.latent_dim * 4
    compressed = model_bytes + latent_bytes
    original = dataset.num_points * dataset.num_timesteps * dataset.num_vars * 4
    return {
        'total_params': total_params,
        'model_size_kb': model_bytes / 1024,
        'latent_size_kb': latent_bytes / 1024,
        'compressed_size_kb': compressed / 1024,
        'original_size_mb': original / (1024 ** 2),
        'compression_ratio': original / compressed,
    }


# Training

def train_conv_autoencoder(model, train_loader, dataset, device,
                           num_epochs, model_name, output_dir):
    """
    Train the convolutional autoencoder.

    Computes metrics on grid-level data during training.
    After training, saves model, latent codes, normalization params, and metrics.

    Returns:
        dict: Per-epoch training metrics.
    """
    os.makedirs(output_dir, exist_ok=True)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    metrics = {
        'loss': [],
        'psnr': [],
        'ssim': [],
        'relative_error': [],
        'time_per_epoch': [],
    }

    comp = compute_compression_ratio(model, dataset)
    total_params = comp['total_params']

    print("")
    print("=" * 65)
    print("  Training: {}".format(model_name))
    print("=" * 65)
    print("  Timesteps (samples) : {}".format(dataset.num_timesteps))
    print("  Grid size           : {} x {}".format(dataset.grid_h, dataset.grid_w))
    print("  Input shape         : ({}, {}, {})".format(
        dataset.num_vars, dataset.grid_h, dataset.grid_w))
    print("  Latent dimension    : {}".format(model.latent_dim))
    print("  Parameters          : {:,}".format(total_params))
    print("  Model size          : {:.2f} KB".format(comp['model_size_kb']))
    print("  Latent storage      : {:.2f} KB ({} x {} x 4 bytes)".format(
        comp['latent_size_kb'], dataset.num_timesteps, model.latent_dim))
    print("  Total compressed    : {:.2f} KB".format(comp['compressed_size_kb']))
    print("  Compression ratio   : {:.1f}:1".format(comp['compression_ratio']))
    print("  Epochs              : {}".format(num_epochs))
    print("  Device              : {}".format(device))
    print("=" * 65)
    print("")

    for epoch in range(num_epochs):
        epoch_start = time.time()
        model.train()
        epoch_loss = 0.0
        all_preds = []
        all_tgts = []

        for inputs, targets in train_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()
            outputs, _ = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            all_preds.append(outputs.detach())
            all_tgts.append(targets)

        epoch_loss /= len(train_loader)
        metrics['loss'].append(epoch_loss)

        all_preds = torch.cat(all_preds, dim=0)
        all_tgts = torch.cat(all_tgts, dim=0)

        # Flatten grid for metric computation
        preds_flat = all_preds.view(all_preds.size(0), -1)
        tgts_flat = all_tgts.view(all_tgts.size(0), -1)

        psnr, ssim = compute_psnr_ssim(preds_flat, tgts_flat, device)
        rel_error = compute_relative_error(preds_flat, tgts_flat)
        metrics['psnr'].append(psnr)
        metrics['ssim'].append(ssim)
        metrics['relative_error'].append(rel_error)

        epoch_time = time.time() - epoch_start
        metrics['time_per_epoch'].append(epoch_time)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(
                "Epoch {:>3d}/{:d}  |  Loss: {:.6f}  |  "
                "PSNR: {:.2f} dB  |  SSIM: {:.4f}  |  "
                "RE: {:.2f}%  |  Time: {:.2f}s".format(
                    epoch + 1, num_epochs, epoch_loss,
                    psnr, ssim, rel_error, epoch_time
                )
            )

    # Save model weights
    model_path = os.path.join(output_dir, "{}.pth".format(model_name))
    torch.save(model.state_dict(), model_path)
    print("")
    print("Model saved to: {}".format(model_path))

    # Save latent codes for all timesteps
    model.eval()
    with torch.no_grad():
        latent_codes = model.encode(dataset.grid_data.to(device)).cpu()
    latent_path = os.path.join(output_dir, "{}_latent_codes.pt".format(model_name))
    torch.save(latent_codes, latent_path)
    print("Latent codes saved to: {} (shape: {})".format(
        latent_path, list(latent_codes.shape)))

    # Save normalization parameters
    norm_path = os.path.join(output_dir, "{}_normalization.json".format(model_name))
    with open(norm_path, 'w') as f:
        json.dump(dataset.get_normalization_params(), f, indent=2)
    print("Normalization parameters saved to: {}".format(norm_path))

    # Save per-epoch metrics
    metrics_path = os.path.join(output_dir, "{}_metrics.csv".format(model_name))
    df = pd.DataFrame(metrics)
    df['epoch'] = range(1, len(df) + 1)
    df = df[['epoch', 'loss', 'psnr', 'ssim', 'relative_error', 'time_per_epoch']]
    df.to_csv(metrics_path, index=False)
    print("Metrics CSV saved to: {}".format(metrics_path))

    total_time = sum(metrics['time_per_epoch'])
    print("")
    print("-" * 65)
    print("  Training completed for: {}".format(model_name))
    print("-" * 65)
    print("  Final Loss           : {:.6f}".format(metrics['loss'][-1]))
    print("  Final PSNR (grid)    : {:.2f} dB".format(metrics['psnr'][-1]))
    print("  Final SSIM (grid)    : {:.4f}".format(metrics['ssim'][-1]))
    print("  Final Rel. Error     : {:.2f}%".format(metrics['relative_error'][-1]))
    print("  Total training time  : {:.1f}s".format(total_time))
    print("-" * 65)
    print("")

    return metrics


print("All utilities loaded successfully.")

In [ ]:
# Configuration

DATA_FILE = "/kaggle/input/ml-test-loader-original-data-csv/ML_test_loader_original_data.csv"
EPOCHS = 150
BATCH_SIZE = 16  # 300 samples of (4, 32, 128) grids: smaller batch for memory
GRID_H = 32
GRID_W = 128
VIS_TIMESTEP_IDX = 0

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Device        : {}".format(device))
print("Epochs        : {}".format(EPOCHS))
print("Batch size    : {}".format(BATCH_SIZE))
print("Grid          : {} x {}".format(GRID_H, GRID_W))

# Load and interpolate dataset
dataset = GridImageDataset(DATA_FILE, grid_h=GRID_H, grid_w=GRID_W)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

input_shape = (GRID_H, GRID_W)

print("")
print("Visualisation timestep index : {}".format(VIS_TIMESTEP_IDX))
print("Visualisation timestep value : {:.4f}".format(
    dataset.time_values[VIS_TIMESTEP_IDX]))

---## 1. Base Conv AE (latent dim = 32): Training

In [ ]:
BASE_DIR = "/kaggle/working/results/conv_ae/base"

base_model = create_conv_autoencoder('base', input_shape).to(device)

base_metrics = train_conv_autoencoder(
    model=base_model,
    train_loader=train_loader,
    dataset=dataset,
    device=device,
    num_epochs=EPOCHS,
    model_name='base_conv_ae',
    output_dir=BASE_DIR,
)

In [ ]:
# Base Conv AE (latent dim = 32): Training Progress Plots

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
epochs_range = range(1, len(base_metrics['loss']) + 1)

axes[0, 0].plot(epochs_range, base_metrics['loss'], 'b-', linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('MSE Loss')
axes[0, 0].set_title('Training Loss')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(epochs_range, base_metrics['psnr'], 'g-', linewidth=2)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('PSNR (dB)')
axes[0, 1].set_title('Peak Signal-to-Noise Ratio')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(epochs_range, base_metrics['ssim'], 'purple', linewidth=2)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('SSIM')
axes[1, 0].set_title('Structural Similarity Index')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim([0, 1.05])

axes[1, 1].plot(epochs_range, base_metrics['relative_error'], 'r-', linewidth=2)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Relative Error (%)')
axes[1, 1].set_title('Reconstruction Error')
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle(
    'Base Conv AE (latent dim = 32): Training Progress',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    os.path.join(BASE_DIR, 'base_conv_ae_training_progress.png'),
    dpi=300, bbox_inches='tight'
)
plt.show()

### 1.1 Base Conv AE (latent dim = 32): Evaluation and Flow-Field Visualisation

In [ ]:
base_model.eval()

total_params = sum(p.numel() for p in base_model.parameters())
print("Evaluating Base Conv AE (latent dim = 32) ({:,} parameters)".format(total_params))

# Grid-level reconstruction
with torch.no_grad():
    base_recon_grid, base_latent = base_model(dataset.grid_data.to(device))
    base_recon_grid = base_recon_grid.cpu()
    base_latent = base_latent.cpu()

# Interpolate predictions back to original mesh points
print("Interpolating grid predictions back to mesh...")
base_recon_mesh = dataset.grid_to_mesh(base_recon_grid.numpy())
base_recon_mesh = np.clip(base_recon_mesh, 0, 1)

# Mesh-level metrics (fair comparison with INR and Linear AE)
mesh_pred_flat = torch.FloatTensor(base_recon_mesh.reshape(-1, dataset.num_vars))
mesh_tgt_flat = torch.FloatTensor(
    dataset.mesh_fields_norm.reshape(-1, dataset.num_vars)
)

base_psnr, base_ssim = compute_psnr_ssim(mesh_pred_flat, mesh_tgt_flat, device)
base_rel_error = compute_relative_error(mesh_pred_flat, mesh_tgt_flat)

print("Mesh-level metrics (fair comparison):")
print("  PSNR           : {:.2f} dB".format(base_psnr))
print("  SSIM           : {:.4f}".format(base_ssim))
print("  Relative Error : {:.2f}%".format(base_rel_error))

# Extract single timestep for visualization
t_idx = VIS_TIMESTEP_IDX
base_pred_t = base_recon_mesh[t_idx]
target_t = dataset.mesh_fields_norm[t_idx]
base_errors_t = np.abs(target_t - base_pred_t)
x = dataset.mesh_coords[:, 0]
y = dataset.mesh_coords[:, 1]

print("")
print("Visualising {:,} points at timestep index {} (t = {:.4f})".format(
    len(x), t_idx, dataset.time_values[t_idx]))

In [ ]:
BASE_DIR = "/kaggle/working/results/conv_ae/base"

# Base Conv AE (latent dim = 32): Flow-Field Visualisation

feature_names = ['Vx', 'Vy', 'Pressure', 'TKE']

fig = plt.figure(figsize=(20, 20))
gs = GridSpec(4, 5, figure=fig,
              width_ratios=[1, 1, 0.05, 1, 0.05],
              wspace=0.35, hspace=0.25)

for row, name in enumerate(feature_names):
    original = target_t[:, row]
    predicted = base_pred_t[:, row]
    error = base_errors_t[:, row]

    ax0 = fig.add_subplot(gs[row, 0])
    ax0.scatter(x, y, c=original, cmap='jet', s=0.5, alpha=0.8, vmin=0, vmax=1)
    ax0.set_title('Original: {}'.format(name))
    ax0.set_aspect('equal')
    ax0.grid(True, alpha=0.3)

    ax1 = fig.add_subplot(gs[row, 1])
    sc2 = ax1.scatter(x, y, c=predicted, cmap='jet', s=0.5, alpha=0.8, vmin=0, vmax=1)
    ax1.set_title('Predicted: {}'.format(name))
    ax1.set_aspect('equal')
    ax1.grid(True, alpha=0.3)

    cax1 = fig.add_subplot(gs[row, 2])
    fig.colorbar(sc2, cax=cax1)

    ax2 = fig.add_subplot(gs[row, 3])
    sc3 = ax2.scatter(x, y, c=error, cmap='hot', s=0.5, alpha=0.8, vmin=0, vmax=1)
    ax2.set_title('Absolute Error: {}'.format(name))
    ax2.set_aspect('equal')
    ax2.grid(True, alpha=0.3)

    cax2 = fig.add_subplot(gs[row, 4])
    fig.colorbar(sc3, cax=cax2)

fig.suptitle(
    'Base Conv AE (latent dim = 32): PSNR: {:.2f} dB  |  t = {:.4f}'.format(
        base_psnr, dataset.time_values[t_idx]),
    fontsize=14, fontweight='bold', y=0.98
)
plt.savefig(
    os.path.join(BASE_DIR, 'base_conv_ae_flow_visualization.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()

print("Base Conv AE (latent dim = 32) evaluation completed.")

---## 2. Medium Conv AE (latent dim = 64): Training

In [ ]:
MEDIUM_DIR = "/kaggle/working/results/conv_ae/medium"

medium_model = create_conv_autoencoder('medium', input_shape).to(device)

medium_metrics = train_conv_autoencoder(
    model=medium_model,
    train_loader=train_loader,
    dataset=dataset,
    device=device,
    num_epochs=EPOCHS,
    model_name='medium_conv_ae',
    output_dir=MEDIUM_DIR,
)

In [ ]:
# Medium Conv AE (latent dim = 64): Training Progress Plots

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
epochs_range = range(1, len(medium_metrics['loss']) + 1)

axes[0, 0].plot(epochs_range, medium_metrics['loss'], 'b-', linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('MSE Loss')
axes[0, 0].set_title('Training Loss')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(epochs_range, medium_metrics['psnr'], 'g-', linewidth=2)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('PSNR (dB)')
axes[0, 1].set_title('Peak Signal-to-Noise Ratio')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(epochs_range, medium_metrics['ssim'], 'purple', linewidth=2)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('SSIM')
axes[1, 0].set_title('Structural Similarity Index')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim([0, 1.05])

axes[1, 1].plot(epochs_range, medium_metrics['relative_error'], 'r-', linewidth=2)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Relative Error (%)')
axes[1, 1].set_title('Reconstruction Error')
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle(
    'Medium Conv AE (latent dim = 64): Training Progress',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    os.path.join(MEDIUM_DIR, 'medium_conv_ae_training_progress.png'),
    dpi=300, bbox_inches='tight'
)
plt.show()

### 2.1 Medium Conv AE (latent dim = 64): Evaluation and Flow-Field Visualisation

In [ ]:
medium_model.eval()

total_params = sum(p.numel() for p in medium_model.parameters())
print("Evaluating Medium Conv AE (latent dim = 64) ({:,} parameters)".format(total_params))

# Grid-level reconstruction
with torch.no_grad():
    medium_recon_grid, medium_latent = medium_model(dataset.grid_data.to(device))
    medium_recon_grid = medium_recon_grid.cpu()
    medium_latent = medium_latent.cpu()

# Interpolate predictions back to original mesh points
print("Interpolating grid predictions back to mesh...")
medium_recon_mesh = dataset.grid_to_mesh(medium_recon_grid.numpy())
medium_recon_mesh = np.clip(medium_recon_mesh, 0, 1)

# Mesh-level metrics (fair comparison with INR and Linear AE)
mesh_pred_flat = torch.FloatTensor(medium_recon_mesh.reshape(-1, dataset.num_vars))
mesh_tgt_flat = torch.FloatTensor(
    dataset.mesh_fields_norm.reshape(-1, dataset.num_vars)
)

medium_psnr, medium_ssim = compute_psnr_ssim(mesh_pred_flat, mesh_tgt_flat, device)
medium_rel_error = compute_relative_error(mesh_pred_flat, mesh_tgt_flat)

print("Mesh-level metrics (fair comparison):")
print("  PSNR           : {:.2f} dB".format(medium_psnr))
print("  SSIM           : {:.4f}".format(medium_ssim))
print("  Relative Error : {:.2f}%".format(medium_rel_error))

# Extract single timestep for visualization
t_idx = VIS_TIMESTEP_IDX
medium_pred_t = medium_recon_mesh[t_idx]
target_t = dataset.mesh_fields_norm[t_idx]
medium_errors_t = np.abs(target_t - medium_pred_t)
x = dataset.mesh_coords[:, 0]
y = dataset.mesh_coords[:, 1]

print("")
print("Visualising {:,} points at timestep index {} (t = {:.4f})".format(
    len(x), t_idx, dataset.time_values[t_idx]))

In [ ]:
MEDIUM_DIR = "/kaggle/working/results/conv_ae/medium"

# Medium Conv AE (latent dim = 64): Flow-Field Visualisation

feature_names = ['Vx', 'Vy', 'Pressure', 'TKE']

fig = plt.figure(figsize=(20, 20))
gs = GridSpec(4, 5, figure=fig,
              width_ratios=[1, 1, 0.05, 1, 0.05],
              wspace=0.35, hspace=0.25)

for row, name in enumerate(feature_names):
    original = target_t[:, row]
    predicted = medium_pred_t[:, row]
    error = medium_errors_t[:, row]

    ax0 = fig.add_subplot(gs[row, 0])
    ax0.scatter(x, y, c=original, cmap='jet', s=0.5, alpha=0.8, vmin=0, vmax=1)
    ax0.set_title('Original: {}'.format(name))
    ax0.set_aspect('equal')
    ax0.grid(True, alpha=0.3)

    ax1 = fig.add_subplot(gs[row, 1])
    sc2 = ax1.scatter(x, y, c=predicted, cmap='jet', s=0.5, alpha=0.8, vmin=0, vmax=1)
    ax1.set_title('Predicted: {}'.format(name))
    ax1.set_aspect('equal')
    ax1.grid(True, alpha=0.3)

    cax1 = fig.add_subplot(gs[row, 2])
    fig.colorbar(sc2, cax=cax1)

    ax2 = fig.add_subplot(gs[row, 3])
    sc3 = ax2.scatter(x, y, c=error, cmap='hot', s=0.5, alpha=0.8, vmin=0, vmax=1)
    ax2.set_title('Absolute Error: {}'.format(name))
    ax2.set_aspect('equal')
    ax2.grid(True, alpha=0.3)

    cax2 = fig.add_subplot(gs[row, 4])
    fig.colorbar(sc3, cax=cax2)

fig.suptitle(
    'Medium Conv AE (latent dim = 64): PSNR: {:.2f} dB  |  t = {:.4f}'.format(
        medium_psnr, dataset.time_values[t_idx]),
    fontsize=14, fontweight='bold', y=0.98
)
plt.savefig(
    os.path.join(MEDIUM_DIR, 'medium_conv_ae_flow_visualization.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()

print("Medium Conv AE (latent dim = 64) evaluation completed.")

---## 3. Large Conv AE (latent dim = 128): Training

In [ ]:
LARGE_DIR = "/kaggle/working/results/conv_ae/large"

large_model = create_conv_autoencoder('large', input_shape).to(device)

large_metrics = train_conv_autoencoder(
    model=large_model,
    train_loader=train_loader,
    dataset=dataset,
    device=device,
    num_epochs=EPOCHS,
    model_name='large_conv_ae',
    output_dir=LARGE_DIR,
)

In [ ]:
# Large Conv AE (latent dim = 128): Training Progress Plots

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
epochs_range = range(1, len(large_metrics['loss']) + 1)

axes[0, 0].plot(epochs_range, large_metrics['loss'], 'b-', linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('MSE Loss')
axes[0, 0].set_title('Training Loss')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(epochs_range, large_metrics['psnr'], 'g-', linewidth=2)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('PSNR (dB)')
axes[0, 1].set_title('Peak Signal-to-Noise Ratio')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(epochs_range, large_metrics['ssim'], 'purple', linewidth=2)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('SSIM')
axes[1, 0].set_title('Structural Similarity Index')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim([0, 1.05])

axes[1, 1].plot(epochs_range, large_metrics['relative_error'], 'r-', linewidth=2)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Relative Error (%)')
axes[1, 1].set_title('Reconstruction Error')
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle(
    'Large Conv AE (latent dim = 128): Training Progress',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    os.path.join(LARGE_DIR, 'large_conv_ae_training_progress.png'),
    dpi=300, bbox_inches='tight'
)
plt.show()

### 3.1 Large Conv AE (latent dim = 128): Evaluation and Flow-Field Visualisation

In [ ]:
large_model.eval()

total_params = sum(p.numel() for p in large_model.parameters())
print("Evaluating Large Conv AE (latent dim = 128) ({:,} parameters)".format(total_params))

# Grid-level reconstruction
with torch.no_grad():
    large_recon_grid, large_latent = large_model(dataset.grid_data.to(device))
    large_recon_grid = large_recon_grid.cpu()
    large_latent = large_latent.cpu()

# Interpolate predictions back to original mesh points
print("Interpolating grid predictions back to mesh...")
large_recon_mesh = dataset.grid_to_mesh(large_recon_grid.numpy())
large_recon_mesh = np.clip(large_recon_mesh, 0, 1)

# Mesh-level metrics (fair comparison with INR and Linear AE)
mesh_pred_flat = torch.FloatTensor(large_recon_mesh.reshape(-1, dataset.num_vars))
mesh_tgt_flat = torch.FloatTensor(
    dataset.mesh_fields_norm.reshape(-1, dataset.num_vars)
)

large_psnr, large_ssim = compute_psnr_ssim(mesh_pred_flat, mesh_tgt_flat, device)
large_rel_error = compute_relative_error(mesh_pred_flat, mesh_tgt_flat)

print("Mesh-level metrics (fair comparison):")
print("  PSNR           : {:.2f} dB".format(large_psnr))
print("  SSIM           : {:.4f}".format(large_ssim))
print("  Relative Error : {:.2f}%".format(large_rel_error))

# Extract single timestep for visualization
t_idx = VIS_TIMESTEP_IDX
large_pred_t = large_recon_mesh[t_idx]
target_t = dataset.mesh_fields_norm[t_idx]
large_errors_t = np.abs(target_t - large_pred_t)
x = dataset.mesh_coords[:, 0]
y = dataset.mesh_coords[:, 1]

print("")
print("Visualising {:,} points at timestep index {} (t = {:.4f})".format(
    len(x), t_idx, dataset.time_values[t_idx]))

In [ ]:
LARGE_DIR = "/kaggle/working/results/conv_ae/large"

# Large Conv AE (latent dim = 128): Flow-Field Visualisation

feature_names = ['Vx', 'Vy', 'Pressure', 'TKE']

fig = plt.figure(figsize=(20, 20))
gs = GridSpec(4, 5, figure=fig,
              width_ratios=[1, 1, 0.05, 1, 0.05],
              wspace=0.35, hspace=0.25)

for row, name in enumerate(feature_names):
    original = target_t[:, row]
    predicted = large_pred_t[:, row]
    error = large_errors_t[:, row]

    ax0 = fig.add_subplot(gs[row, 0])
    ax0.scatter(x, y, c=original, cmap='jet', s=0.5, alpha=0.8, vmin=0, vmax=1)
    ax0.set_title('Original: {}'.format(name))
    ax0.set_aspect('equal')
    ax0.grid(True, alpha=0.3)

    ax1 = fig.add_subplot(gs[row, 1])
    sc2 = ax1.scatter(x, y, c=predicted, cmap='jet', s=0.5, alpha=0.8, vmin=0, vmax=1)
    ax1.set_title('Predicted: {}'.format(name))
    ax1.set_aspect('equal')
    ax1.grid(True, alpha=0.3)

    cax1 = fig.add_subplot(gs[row, 2])
    fig.colorbar(sc2, cax=cax1)

    ax2 = fig.add_subplot(gs[row, 3])
    sc3 = ax2.scatter(x, y, c=error, cmap='hot', s=0.5, alpha=0.8, vmin=0, vmax=1)
    ax2.set_title('Absolute Error: {}'.format(name))
    ax2.set_aspect('equal')
    ax2.grid(True, alpha=0.3)

    cax2 = fig.add_subplot(gs[row, 4])
    fig.colorbar(sc3, cax=cax2)

fig.suptitle(
    'Large Conv AE (latent dim = 128): PSNR: {:.2f} dB  |  t = {:.4f}'.format(
        large_psnr, dataset.time_values[t_idx]),
    fontsize=14, fontweight='bold', y=0.98
)
plt.savefig(
    os.path.join(LARGE_DIR, 'large_conv_ae_flow_visualization.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()

print("Large Conv AE (latent dim = 128) evaluation completed.")

---## 4. Combined Comparison: All Three Models

In [ ]:
# Training Curves Comparison

COMP_DIR = "/kaggle/working/results/conv_ae/comparison"
os.makedirs(COMP_DIR, exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

models_data = [
    ('Base (latent = 32)',   base_metrics,   'tab:blue'),
    ('Medium (latent = 64)', medium_metrics, 'tab:orange'),
    ('Large (latent = 128)', large_metrics,  'tab:green'),
]

for label, m, color in models_data:
    er = range(1, len(m['loss']) + 1)
    axes[0, 0].plot(er, m['loss'],           '-', color=color, linewidth=2, label=label)
    axes[0, 1].plot(er, m['psnr'],           '-', color=color, linewidth=2, label=label)
    axes[1, 0].plot(er, m['ssim'],           '-', color=color, linewidth=2, label=label)
    axes[1, 1].plot(er, m['relative_error'], '-', color=color, linewidth=2, label=label)

axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('MSE Loss')
axes[0, 0].set_title('Training Loss')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('PSNR (dB)')
axes[0, 1].set_title('Peak Signal-to-Noise Ratio')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('SSIM')
axes[1, 0].set_title('Structural Similarity Index')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim([0, 1.05])
axes[1, 0].legend()

axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Relative Error (%)')
axes[1, 1].set_title('Reconstruction Error')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

plt.suptitle(
    'Conv AutoEncoder: Model Comparison (Training, grid-level)',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    os.path.join(COMP_DIR, 'conv_ae_training_comparison.png'),
    dpi=300, bbox_inches='tight'
)
plt.show()

In [ ]:
# Evaluation Metrics: Bar Charts (mesh-level)

model_labels = [
    'Base\n(latent = 32)',
    'Medium\n(latent = 64)',
    'Large\n(latent = 128)',
]
eval_psnrs  = [base_psnr,      medium_psnr,      large_psnr]
eval_ssims  = [base_ssim,      medium_ssim,      large_ssim]
eval_errors = [base_rel_error, medium_rel_error, large_rel_error]
colors = ['tab:blue', 'tab:orange', 'tab:green']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

bars1 = axes[0].bar(model_labels, eval_psnrs, color=colors,
                     edgecolor='black', linewidth=0.5)
axes[0].set_ylabel('PSNR (dB)')
axes[0].set_title('Evaluation PSNR (mesh-level)')
axes[0].grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars1, eval_psnrs):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
        '{:.2f}'.format(val), ha='center', va='bottom', fontsize=10
    )

bars2 = axes[1].bar(model_labels, eval_ssims, color=colors,
                     edgecolor='black', linewidth=0.5)
axes[1].set_ylabel('SSIM')
axes[1].set_title('Evaluation SSIM (mesh-level)')
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_ylim([0, 1.1])
for bar, val in zip(bars2, eval_ssims):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
        '{:.4f}'.format(val), ha='center', va='bottom', fontsize=10
    )

bars3 = axes[2].bar(model_labels, eval_errors, color=colors,
                     edgecolor='black', linewidth=0.5)
axes[2].set_ylabel('Relative Error (%)')
axes[2].set_title('Evaluation Relative Error (mesh-level)')
axes[2].grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars3, eval_errors):
    axes[2].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
        '{:.2f}%'.format(val), ha='center', va='bottom', fontsize=10
    )

plt.suptitle(
    'Conv AutoEncoder: Evaluation Metrics Comparison (mesh-level)',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    os.path.join(COMP_DIR, 'conv_ae_evaluation_comparison.png'),
    dpi=300, bbox_inches='tight'
)
plt.show()

In [ ]:
# Compression Analysis: Storage Breakdown and Ratios

comp_base   = compute_compression_ratio(base_model, dataset)
comp_medium = compute_compression_ratio(medium_model, dataset)
comp_large  = compute_compression_ratio(large_model, dataset)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_sizes_mb = [
    comp_base['model_size_kb'] / 1024,
    comp_medium['model_size_kb'] / 1024,
    comp_large['model_size_kb'] / 1024,
]
latent_sizes_mb = [
    comp_base['latent_size_kb'] / 1024,
    comp_medium['latent_size_kb'] / 1024,
    comp_large['latent_size_kb'] / 1024,
]
x_pos = range(len(model_labels))

axes[0].bar(
    x_pos, model_sizes_mb, color='steelblue',
    label='Model Weights', edgecolor='black', linewidth=0.5
)
axes[0].bar(
    x_pos, latent_sizes_mb, bottom=model_sizes_mb, color='coral',
    label='Latent Codes', edgecolor='black', linewidth=0.5
)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(model_labels)
axes[0].set_ylabel('Size (MB)')
axes[0].set_title('Compressed Storage Breakdown')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')
for i, (ms, ls) in enumerate(zip(model_sizes_mb, latent_sizes_mb)):
    axes[0].text(
        i, ms + ls + 0.05, '{:.2f} MB'.format(ms + ls),
        ha='center', fontsize=10
    )

ratios = [
    comp_base['compression_ratio'],
    comp_medium['compression_ratio'],
    comp_large['compression_ratio'],
]
bars_r = axes[1].bar(
    model_labels, ratios, color=colors,
    edgecolor='black', linewidth=0.5
)
axes[1].set_ylabel('Compression Ratio')
axes[1].set_title('Compression Ratio (higher is better)')
axes[1].grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars_r, ratios):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
        '{:.1f}:1'.format(val), ha='center', va='bottom', fontsize=10
    )

plt.suptitle(
    'Conv AutoEncoder: Compression Analysis',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    os.path.join(COMP_DIR, 'conv_ae_compression_comparison.png'),
    dpi=300, bbox_inches='tight'
)
plt.show()

---## 5. Per-Timestep Reconstruction Quality

In [ ]:
# Per-Timestep PSNR and Relative Error (mesh-level)

print("Computing per-timestep mesh-level metrics...")

timestep_psnrs  = {'base': [], 'medium': [], 'large': []}
timestep_errors = {'base': [], 'medium': [], 'large': []}

recon_map = {
    'base':   base_recon_mesh,
    'medium': medium_recon_mesh,
    'large':  large_recon_mesh,
}

for t_i in range(dataset.num_timesteps):
    target_ti = torch.FloatTensor(dataset.mesh_fields_norm[t_i])

    for name in ['base', 'medium', 'large']:
        pred_ti = torch.FloatTensor(recon_map[name][t_i])

        psnr_metric = PeakSignalNoiseRatio()
        psnr_metric.update(pred_ti, target_ti)
        timestep_psnrs[name].append(psnr_metric.compute().item())

        err = torch.norm(pred_ti - target_ti) / torch.norm(target_ti) * 100
        timestep_errors[name].append(err.item())

print("Done. Computed metrics across {} timesteps.".format(dataset.num_timesteps))

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

timesteps = range(dataset.num_timesteps)
plot_info = [
    ('base',   'Base (latent = 32)',   'tab:blue'),
    ('medium', 'Medium (latent = 64)', 'tab:orange'),
    ('large',  'Large (latent = 128)', 'tab:green'),
]

for key, label, color in plot_info:
    axes[0].plot(
        timesteps, timestep_psnrs[key], '-',
        color=color, linewidth=1.5, label=label, alpha=0.8
    )
    axes[1].plot(
        timesteps, timestep_errors[key], '-',
        color=color, linewidth=1.5, label=label, alpha=0.8
    )

axes[0].set_xlabel('Timestep Index')
axes[0].set_ylabel('PSNR (dB)')
axes[0].set_title('Per-Timestep PSNR (mesh-level)')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].set_xlabel('Timestep Index')
axes[1].set_ylabel('Relative Error (%)')
axes[1].set_title('Per-Timestep Relative Error (mesh-level)')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.suptitle(
    'Conv AutoEncoder: Reconstruction Quality Across Timesteps',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    os.path.join(COMP_DIR, 'conv_ae_per_timestep_quality.png'),
    dpi=300, bbox_inches='tight'
)
plt.show()

---## 6. Summary

In [ ]:
# Final Summary Table

header = "{:<30s} {:>18s} {:>20s} {:>20s}".format(
    "Metric", "Base (latent=32)", "Medium (latent=64)", "Large (latent=128)"
)

print("")
print("=" * 90)
print("  CONV AUTOENCODER: THREE-MODEL COMPARISON SUMMARY")
print("=" * 90)
print(header)
print("-" * 90)

print("{:<30s} {:>18,d} {:>20,d} {:>20,d}".format(
    "Parameters",
    comp_base['total_params'],
    comp_medium['total_params'],
    comp_large['total_params'],
))
print("{:<30s} {:>18.1f} {:>20.1f} {:>20.1f}".format(
    "Model Size (KB)",
    comp_base['model_size_kb'],
    comp_medium['model_size_kb'],
    comp_large['model_size_kb'],
))
print("{:<30s} {:>18.2f} {:>20.2f} {:>20.2f}".format(
    "Latent Codes (KB)",
    comp_base['latent_size_kb'],
    comp_medium['latent_size_kb'],
    comp_large['latent_size_kb'],
))
print("{:<30s} {:>18.1f} {:>20.1f} {:>20.1f}".format(
    "Total Compressed (KB)",
    comp_base['compressed_size_kb'],
    comp_medium['compressed_size_kb'],
    comp_large['compressed_size_kb'],
))
print("{:<30s} {:>17.1f}:1 {:>19.1f}:1 {:>19.1f}:1".format(
    "Compression Ratio",
    comp_base['compression_ratio'],
    comp_medium['compression_ratio'],
    comp_large['compression_ratio'],
))

print("-" * 90)

print("{:<30s} {:>18.2f} {:>20.2f} {:>20.2f}".format(
    "Eval PSNR (dB, mesh)",
    base_psnr, medium_psnr, large_psnr,
))
print("{:<30s} {:>18.4f} {:>20.4f} {:>20.4f}".format(
    "Eval SSIM (mesh)",
    base_ssim, medium_ssim, large_ssim,
))
print("{:<30s} {:>18.2f} {:>20.2f} {:>20.2f}".format(
    "Eval Rel. Error (%, mesh)",
    base_rel_error, medium_rel_error, large_rel_error,
))
print("{:<30s} {:>18.1f} {:>20.1f} {:>20.1f}".format(
    "Total Training Time (s)",
    sum(base_metrics['time_per_epoch']),
    sum(medium_metrics['time_per_epoch']),
    sum(large_metrics['time_per_epoch']),
))

print("=" * 90)

In [ ]:
# Save All Evaluation Results to JSON

all_results = {
    'base': {
        'architecture': 'Conv2D [16,32,64,128] latent=32',
        'latent_dim': 32,
        'grid': '{}x{}'.format(GRID_H, GRID_W),
        'parameters': comp_base['total_params'],
        'model_size_kb': comp_base['model_size_kb'],
        'latent_size_kb': comp_base['latent_size_kb'],
        'compressed_size_kb': comp_base['compressed_size_kb'],
        'compression_ratio': comp_base['compression_ratio'],
        'eval_psnr_mesh': float(base_psnr),
        'eval_ssim_mesh': float(base_ssim),
        'eval_relative_error_mesh': float(base_rel_error),
        'training_time_s': sum(base_metrics['time_per_epoch']),
    },
    'medium': {
        'architecture': 'Conv2D [32,64,128,256] latent=64',
        'latent_dim': 64,
        'grid': '{}x{}'.format(GRID_H, GRID_W),
        'parameters': comp_medium['total_params'],
        'model_size_kb': comp_medium['model_size_kb'],
        'latent_size_kb': comp_medium['latent_size_kb'],
        'compressed_size_kb': comp_medium['compressed_size_kb'],
        'compression_ratio': comp_medium['compression_ratio'],
        'eval_psnr_mesh': float(medium_psnr),
        'eval_ssim_mesh': float(medium_ssim),
        'eval_relative_error_mesh': float(medium_rel_error),
        'training_time_s': sum(medium_metrics['time_per_epoch']),
    },
    'large': {
        'architecture': 'Conv2D [32,64,128,256] latent=128',
        'latent_dim': 128,
        'grid': '{}x{}'.format(GRID_H, GRID_W),
        'parameters': comp_large['total_params'],
        'model_size_kb': comp_large['model_size_kb'],
        'latent_size_kb': comp_large['latent_size_kb'],
        'compressed_size_kb': comp_large['compressed_size_kb'],
        'compression_ratio': comp_large['compression_ratio'],
        'eval_psnr_mesh': float(large_psnr),
        'eval_ssim_mesh': float(large_ssim),
        'eval_relative_error_mesh': float(large_rel_error),
        'training_time_s': sum(large_metrics['time_per_epoch']),
    },
}

results_path = os.path.join(COMP_DIR, 'conv_ae_all_results.json')
with open(results_path, 'w') as f:
    json.dump(all_results, f, indent=2)

print("All results saved to: {}".format(results_path))

---## 7. Download Results

In [ ]:
# Package all results into a single zip for download

import shutil

zip_path = shutil.make_archive(
    '/kaggle/working/all_results',
    'zip',
    '/kaggle/working/results'
)

zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)

print("Results archived to: {}".format(zip_path))
print("Archive size: {:.1f} MB".format(zip_size_mb))
print("")
print("To download:")
print("  1. Go to the Output tab on the right panel")
print("  2. Click on 'all_results.zip' to download")
print("  3. Or download individual files from the results/ folder")